# Qualidade dos dados

**Objetivo:** verificar a consistência dos arquivos gerados no notebook 01 antes de qualquer análise.

O notebook tem quatro partes:

1. **Conferências de soma:** as partes de cada tabela precisam somar o total nacional.
2. **Reconciliação das variações:** os valores de 2024 implícitos nas variações do Anuário 2026 são comparados com os valores publicados no Anuário 2025.
3. **Registro de inconsistências:** cada problema encontrado nas fontes e a decisão tomada.
4. **Dicionário de dados:** descrição de cada arquivo em `docs/dicionario_de_dados.md`.

## Preparação

In [13]:
import pandas as pd
from pathlib import Path

pd.options.display.float_format = "{:,.2f}".format

PASTA_SAIDA = Path("../data/processed")

TOTAL_2025 = 15_688_083_191.69
TOTAL_2024_ORIGINAL = 15_344_065_267.36
TOTAL_2024_RETIFICADO = 17_210_754_610.75

TOLERANCIA_LITROS = 1.0
TOLERANCIA_PCT = 0.1

def ler(nome_arquivo):
    return pd.read_csv(PASTA_SAIDA / nome_arquivo)

## 1. Conferências de soma

Cada conferência compara um valor calculado a partir dos CSVs com o valor esperado, publicado no Anuário. Uma conferência é aprovada quando a diferença fica dentro da tolerância.

In [14]:
conferencias = []

def conferir(descricao, calculado, esperado, tolerancia):
    diferenca = calculado - esperado
    conferencias.append({
        "conferencia": descricao,
        "calculado": calculado,
        "esperado": esperado,
        "diferenca": diferenca,
        "ok": abs(diferenca) <= tolerancia,
    })

In [15]:
t17 = ler("tabela_17_producao_regiao_2025.csv")
t18 = ler("tabela_18_producao_media_regiao_2025.csv")
t20 = ler("tabela_20_puro_malte_2025.csv")
t23 = ler("tabela_23_abrangencia_2025.csv")

conferir("2025 | regiões (T17) = total", t17["volume_litros"].sum(), TOTAL_2025, TOLERANCIA_LITROS)
conferir("2025 | regiões (T18) = total", t18["volume_litros"].sum(), TOTAL_2025, TOLERANCIA_LITROS)
conferir("2025 | puro malte + demais (T20) = total", t20["volume_litros"].sum(), TOTAL_2025, TOLERANCIA_LITROS)
conferir("2025 | abrangência (T23) = total", t23["volume_litros"].sum(), TOTAL_2025, TOLERANCIA_LITROS)

conferir("2024 | regiões = total", ler("producao_regiao_2024.csv")["volume_litros"].sum(), TOTAL_2024_ORIGINAL, TOLERANCIA_LITROS)
conferir("2024 | puro malte + demais = total", ler("puro_malte_2024.csv")["volume_litros"].sum(), TOTAL_2024_ORIGINAL, TOLERANCIA_LITROS)
conferir("2024 | abrangência = total", ler("abrangencia_2024.csv")["volume_litros"].sum(), TOTAL_2024_ORIGINAL, TOLERANCIA_LITROS)
conferir("2024 | estilos = total", ler("estilos_2024.csv")["volume_litros"].sum(), TOTAL_2024_ORIGINAL, TOLERANCIA_LITROS)

conferir("2025 | % de empregos por UF (T16) = 100", ler("tabela_16_empregos_uf_2025.csv")["pct_empregos"].sum(), 100, TOLERANCIA_PCT)
conferir("2025 | % entre estilos (T21) = 100", ler("tabela_21_estilos_2025.csv")["pct_entre_estilos"].sum(), 100, TOLERANCIA_PCT)
conferir("2025 | % de abrangência (T23) = 100", t23["pct_volume"].sum(), 100, TOLERANCIA_PCT)

In [16]:
volume_t17 = t17.set_index("regiao")["volume_litros"]
volume_t18 = t18.set_index("regiao")["volume_litros"]
cervejarias_t18 = t18.set_index("regiao")["n_cervejarias"]
cervejarias_graf2 = ler("grafico_02_cervejarias_uf_2025.csv").groupby("regiao")["n_cervejarias"].sum()

for regiao in volume_t17.index:
    conferir(f"2025 | {regiao}: volume T17 = T18", volume_t17[regiao], volume_t18[regiao], TOLERANCIA_LITROS)
    conferir(f"2025 | {regiao}: cervejarias Gráfico 2 = T18", cervejarias_graf2[regiao], cervejarias_t18[regiao], 0)

media_calculada = t18["volume_litros"] / t18["n_cervejarias"]
maior_erro_media = (media_calculada - t18["producao_media_litros"]).abs().max()
conferir("2025 | produção média (T18) = volume ÷ cervejarias", maior_erro_media, 0, TOLERANCIA_LITROS)

In [17]:
df_conferencias = pd.DataFrame(conferencias)
print("Aprovadas:", df_conferencias["ok"].sum(), "de", len(df_conferencias))
df_conferencias

Aprovadas: 22 de 22


,conferencia,calculado,esperado,diferenca,ok
0,2025 | regiões (T17) = total,"15,688,083,191.69","15,688,083,191.69",-0.00,True
1,2025 | regiões (T18) = total,"15,688,083,191.69","15,688,083,191.69",-0.00,True
2,2025 | puro malte + demais (T20) = total,"15,688,083,191.69","15,688,083,191.69",0.00,True
3,2025 | abrangência (T23) = total,"15,688,083,191.69","15,688,083,191.69",0.00,True
4,2024 | regiões = total,"15,344,065,267.36","15,344,065,267.36",-0.00,True
5,2024 | puro malte + demais = total,"15,344,065,267.36","15,344,065,267.36",-0.00,True
6,2024 | abrangência = total,"15,344,065,267.36","15,344,065,267.36",0.00,True
7,2024 | estilos = total,"15,344,065,267.36","15,344,065,267.36",-0.00,True
8,2025 | % de empregos por UF (T16) = 100,100.00,100.00,0.00,True
9,2025 | % entre estilos (T21) = 100,100.01,100.00,0.01,True


## 2. Reconciliação das variações

O Anuário 2026 retificou o total de 2024 de 15,34 para 17,21 bilhões de litros. Para saber qual base cada variação usou, calculamos o volume de 2024 implícito em cada uma:

**2024 implícito = volume 2025 ÷ (1 + variação / 100)**

e comparamos com o valor publicado no Anuário 2025 (base original). Somando as categorias de cada tabela, vemos qual dos dois totais de 2024 a tabela reproduz.

In [18]:
regiao_2025 = ler("tabela_17_producao_regiao_2025.csv").rename(columns={"regiao": "categoria"})
regiao_2024 = ler("producao_regiao_2024.csv").rename(columns={"regiao": "categoria"})

malte_2025 = ler("tabela_20_puro_malte_2025.csv")
malte_2024 = ler("puro_malte_2024.csv")

abrang_2025 = ler("tabela_23_abrangencia_2025.csv").rename(columns={"abrangencia": "categoria"})
abrang_2024 = ler("abrangencia_2024.csv").rename(columns={"abrangencia": "categoria"})

pares = [
    ("Região (T17)", regiao_2025, regiao_2024),
    ("Puro malte (T20)", malte_2025, malte_2024),
    ("Abrangência (T23)", abrang_2025, abrang_2024),
]

partes = []
for tabela, df_2025, df_2024 in pares:
    juncao = df_2025[["categoria", "volume_litros", "variacao_pct"]].merge(
        df_2024[["categoria", "volume_litros"]],
        on="categoria",
        how="left",
        suffixes=("_2025", "_2024_publicado"),
    )
    juncao.insert(0, "tabela", tabela)
    partes.append(juncao)

recon = pd.concat(partes, ignore_index=True)
print("Categorias sem par em 2024:", recon["volume_litros_2024_publicado"].isna().sum())

Categorias sem par em 2024: 0


In [19]:
recon["volume_2024_implicito"] = recon["volume_litros_2025"] / (1 + recon["variacao_pct"] / 100)
recon["dif_pct_vs_publicado"] = (recon["volume_2024_implicito"] / recon["volume_litros_2024_publicado"] - 1) * 100

recon.to_csv(PASTA_SAIDA / "reconciliacao_variacoes_2024.csv", index=False)

recon[["tabela", "categoria", "variacao_pct", "volume_litros_2024_publicado", "volume_2024_implicito", "dif_pct_vs_publicado"]]

,tabela,categoria,variacao_pct,volume_litros_2024_publicado,volume_2024_implicito,dif_pct_vs_publicado
0,Região (T17),Centro-Oeste,-5.59,"1,047,137,071.79","1,116,670,690.87",6.64
1,Região (T17),Nordeste,-8.07,"3,290,546,306.35","4,135,997,460.15",25.69
2,Região (T17),Norte,-16.92,"221,378,694.16","259,590,421.26",17.26
3,Região (T17),Sudeste,-9.36,"8,432,282,117.12","9,011,952,582.84",6.87
4,Região (T17),Sul,-8.90,"2,352,721,077.94","2,686,619,521.68",14.19
5,Puro malte (T20),Cerveja Puro Malte ou 100% malte,21.01,"3,785,009,947.73","3,785,068,152.96",0.00
6,Puro malte (T20),Demais cervejas,-3.90,"11,559,055,319.63","11,558,555,899.89",-0.00
7,Abrangência (T23),Microrregional (raio até 100 km),-18.34,"34,992,867.48","34,992,071.16",-0.00
8,Abrangência (T23),Regional (raio >100km a 500km),-12.75,"83,539,433.00","83,540,800.91",0.00
9,Abrangência (T23),Nacional (>500 km ou no mínimo duas UF),3.04,"12,814,184,421.93","12,814,447,481.84",0.00


In [20]:
resumo = recon.groupby("tabela", sort=False)["volume_2024_implicito"].sum().to_frame("total_2024_implicito")
resumo["dif_pct_vs_original"] = (resumo["total_2024_implicito"] / TOTAL_2024_ORIGINAL - 1) * 100
resumo["dif_pct_vs_retificado"] = (resumo["total_2024_implicito"] / TOTAL_2024_RETIFICADO - 1) * 100

resumo["base_usada"] = "original"
mais_perto_da_retificada = resumo["dif_pct_vs_retificado"].abs() < resumo["dif_pct_vs_original"].abs()
resumo.loc[mais_perto_da_retificada, "base_usada"] = "retificada"

resumo

,total_2024_implicito,dif_pct_vs_original,dif_pct_vs_retificado,base_usada
tabela,,,,
Região (T17),"17,210,830,676.80",12.17,0.00,retificada
Puro malte (T20),"15,343,624,052.85",-0.00,-10.85,original
Abrangência (T23),"15,344,291,472.05",0.00,-10.84,original


In [21]:
eh_puro_malte_2025 = malte_2025["categoria"].str.startswith("Cerveja Puro Malte")
eh_puro_malte_2024 = malte_2024["categoria"].str.startswith("Cerveja Puro Malte")
puro_malte_2025 = malte_2025.loc[eh_puro_malte_2025, "volume_litros"].iloc[0]
puro_malte_2024 = malte_2024.loc[eh_puro_malte_2024, "volume_litros"].iloc[0]

retificacao = TOTAL_2024_RETIFICADO - TOTAL_2024_ORIGINAL
participacao_2024 = puro_malte_2024 / TOTAL_2024_ORIGINAL

cenarios = pd.DataFrame({
    "cenario": [
        "Base original (como o Anuário 2026 publica)",
        "Retificação distribuída na proporção de 2024",
        "Retificação toda em 'demais cervejas'",
        "Retificação toda em puro malte",
    ],
    "puro_malte_2024": [
        puro_malte_2024,
        puro_malte_2024 + retificacao * participacao_2024,
        puro_malte_2024,
        puro_malte_2024 + retificacao,
    ],
})
cenarios["variacao_pct"] = (puro_malte_2025 / cenarios["puro_malte_2024"] - 1) * 100

print(f"Retificação de 2024: {retificacao:,.0f} litros")
print(f"Participação do puro malte: {participacao_2024:.1%} em 2024 e {puro_malte_2025 / TOTAL_2025:.1%} em 2025")
cenarios

Retificação de 2024: 1,866,689,343 litros
Participação do puro malte: 24.7% em 2024 e 29.2% em 2025


,cenario,puro_malte_2024,variacao_pct
0,Base original (como o Anuário 2026 publica),"3,785,009,947.73",21.01
1,Retificação distribuída na proporção de 2024,"4,245,477,080.20",7.89
2,Retificação toda em 'demais cervejas',"3,785,009,947.73",21.01
3,Retificação toda em puro malte,"5,651,699,291.12",-18.96


## 3. Conferências do comércio exterior

Os totais anuais de cerveja do Comex Stat são comparados com as tabelas 6 e 7 do Anuário 2026, e o superávit publicado no texto é comparado com o calculado a partir dessas tabelas. A tolerância é de 1%, porque o Comex Stat revisa dados passados.

In [22]:
TOLERANCIA_RELATIVA = 0.01

cerveja = ler("comex_cerveja_mensal_2015_2025.csv")
anual = cerveja.groupby(["fluxo", "ano"])[["valor_fob_usd", "litros"]].sum()

anuario_comercio = {
    ("exportacao", 2024): (204_289_191, 332_540_021),
    ("exportacao", 2025): (218_356_084, 315_519_948),
    ("importacao", 2024): (9_258_191, 7_492_350),
    ("importacao", 2025): (9_418_836, 26_327_681),
}

for (fluxo, ano), (valor_anuario, litros_anuario) in anuario_comercio.items():
    conferir(f"{ano} | {fluxo}: valor Comex = Anuário", anual.loc[(fluxo, ano), "valor_fob_usd"], valor_anuario, valor_anuario * TOLERANCIA_RELATIVA)
    conferir(f"{ano} | {fluxo}: litros Comex = Anuário", anual.loc[(fluxo, ano), "litros"], litros_anuario, litros_anuario * TOLERANCIA_RELATIVA)

valor_exportado_2025, _ = anuario_comercio[("exportacao", 2025)]
valor_importado_2025, _ = anuario_comercio[("importacao", 2025)]
conferir("2025 | superávit (Tab. 6 − Tab. 7) = texto da p. 35", valor_exportado_2025 - valor_importado_2025, 195_031_000, 1)

df_conferencias = pd.DataFrame(conferencias)
print("Aprovadas:", df_conferencias["ok"].sum(), "de", len(df_conferencias))
df_conferencias[~df_conferencias["ok"]]

Aprovadas: 30 de 31


,conferencia,calculado,esperado,diferenca,ok
30,2025 | superávit (Tab. 6 − Tab. 7) = texto da ...,"208,937,248.00","195,031,000.00","13,906,248.00",False


## 4. Registro de inconsistências

Cada problema encontrado nas fontes, com a evidência e a decisão tomada. O registro é salvo em `docs/registro_inconsistencias.csv`.

In [23]:
PASTA_DOCS = Path("../docs")

inconsistencias = [
    {
        "fonte": "Anuário 2026",
        "local": "Tabelas 17, 20 e 23",
        "problema": "As variações 2024/2025 usam bases diferentes para 2024: a Tabela 17 usa o total retificado (17,21 bi L); as tabelas 20 e 23, o original (15,34 bi L).",
        "evidencia": "Reconciliação (seção 2): a soma dos volumes de 2024 implícitos reproduz um total diferente em cada tabela.",
        "impacto": "alto",
        "decisao": "Não comparar o +21,01% do puro malte com a queda de 8,85%. Indicador principal: participação do puro malte (24,7% → 29,2%). O +21% é citado só como teto, junto com os cenários.",
    },
    {
        "fonte": "Anuário 2026",
        "local": "Tabela 17 × Anuário 2025",
        "problema": "A retificação de 2024 não foi uniforme: Nordeste +25,7% (cerca de 845 milhões de L), Norte +17,3%, Sul +14,2%, Sudeste +6,9%, Centro-Oeste +6,6%.",
        "evidencia": "Reconciliação (seção 2), coluna dif_pct_vs_publicado.",
        "impacto": "médio",
        "decisao": "Volumes regionais de 2024 publicados no Anuário 2025 não são usados para calcular variações. Comparações regionais entre anos usam participação. A retificação é documentada como achado.",
    },
    {
        "fonte": "Anuário 2025",
        "local": "Tabela 17",
        "problema": "Volume do Centro-Oeste impresso como 1.074.137.071,79 L; a Tabela 18 e o texto informam 1.047.137.071,79 L.",
        "evidencia": "Só com 1.047.137.071,79 L as regiões somam o total nacional.",
        "impacto": "baixo",
        "decisao": "Usado 1.047.137.071,79 L, com observação na coluna `observacao` de producao_regiao_2024.csv.",
    },
    {
        "fonte": "Anuário 2025",
        "local": "Tabela 18",
        "problema": "A coluna Produção Média não corresponde a volume ÷ número de cervejarias em nenhuma região.",
        "evidencia": "A média recalculada coincide com os valores citados no texto da p. 50. Na edição de 2026 a coluna está correta.",
        "impacto": "baixo",
        "decisao": "Coluna impressa mantida; análises usam `producao_media_calculada`.",
    },
    {
        "fonte": "Anuário 2026",
        "local": "Gráfico 2",
        "problema": "A legenda indica variação 2023/2024, mas os valores são de 2024/2025.",
        "evidencia": "São Paulo: 427 cervejarias no Anuário 2025 e 452 no de 2026, alta de 5,9%, igual ao gráfico.",
        "impacto": "baixo",
        "decisao": "Variação tratada como 2024/2025, com observação no CSV.",
    },
    {
        "fonte": "Anuário 2026",
        "local": "Texto do Gráfico 8 (p. 35)",
        "problema": "O superávit de 2025 aparece como US$ 195.031.000, que é o valor de 2024.",
        "evidencia": "Tabela 6 − Tabela 7 = US$ 208.937.248. O crescimento de 7,1% citado no mesmo parágrafo corresponde a esse valor.",
        "impacto": "baixo",
        "decisao": "Superávit de 2025 calculado a partir das tabelas: US$ 208,9 milhões.",
    },
    {
        "fonte": "Anuário 2026",
        "local": "Gráficos 10 e 12",
        "problema": "Títulos trocados. O Gráfico 10 ('importadores de cerveja brasileira') mostra as origens das importações; o Gráfico 12 ('exportadores para o Brasil') mostra os destinos das exportações.",
        "evidencia": "O Gráfico 10 lista Alemanha, Argentina e Uruguai em milhares de US$, como a Tabela 12; o Gráfico 12 lista Paraguai e Bolívia em milhões de US$ e 'Outros (67)', que somados aos 10 dão os 77 destinos da exportação.",
        "impacto": "baixo",
        "decisao": "Gráficos não usados; comércio exterior vem do Comex Stat.",
    },
    {
        "fonte": "Anuário 2026",
        "local": "Texto da Tabela 7 (p. 34) e Tabela 10",
        "problema": "Erros de digitação: volume de 2024 escrito '7.492.3350' e valor de 2025 'US$ 9.418.191' (a tabela traz 9.418.836). O título da Tabela 10 diz 2024, mas os dados são de 2025.",
        "evidencia": "Comparação entre texto, tabelas e Anuário 2025.",
        "impacto": "baixo",
        "decisao": "Valores das tabelas prevalecem sobre o texto.",
    },
    {
        "fonte": "Anuário 2026",
        "local": "Tabelas 7 e 13",
        "problema": "Importação de cerveja em 2025: volume +251% com valor +1,7%; os EUA aparecem com preço médio de 0,01 US$/L.",
        "evidencia": "O Comex Stat confirma os 26,3 milhões de litros (conferência da seção 3), então a anomalia está nas declarações de importação, não no Anuário. Um preço de 0,01 US$/L é implausível e sugere erro na quantidade declarada.",
        "impacto": "médio",
        "decisao": "Investigar na fase 4: localizar nos dados mensais do Comex o mês e o NCM do salto. Até lá, volumes importados de 2025 não são usados.",
    },
    {
        "fonte": "Comex Stat",
        "local": "Definição de cerveja",
        "problema": "O Anuário considera cerveja a soma dos NCM 2203.00.00 e 2202.91.00 (sem álcool), sem documentar isso.",
        "evidencia": "Só com os dois códigos os totais de 2024 batem com o Anuário (diferenças abaixo de 0,02%).",
        "impacto": "médio",
        "decisao": "Cerveja = 2203.00.00 + 2202.91.00. Em 2015 e 2016 só existe o 2203 (o 2202.91 foi criado em 2017).",
    },
    {
        "fonte": "Comex Stat",
        "local": "Cerveja, kg líquido",
        "problema": "Na exportação, o peso declarado (291,5 mi kg em 2024) é menor que o volume (332,5 mi L), o que é fisicamente impossível para cerveja.",
        "evidencia": "Consulta por NCM com quantidade estatística.",
        "impacto": "baixo",
        "decisao": "Para cerveja, usar a quantidade estatística em litros.",
    },
    {
        "fonte": "Anuários 2025 e 2026",
        "local": "Estilos e características",
        "problema": "As categorias mudam entre edições: a de 2026 exclui os estilos de maior volume e agrupa outros; 'Com madeira' (2024) e 'Com Madeiras Brasileiras' (2025) são definições diferentes.",
        "evidencia": "Comparação das listas das duas edições.",
        "impacto": "médio",
        "decisao": "Comparar entre anos só categorias com a mesma definição.",
    },
]

df_inconsistencias = pd.DataFrame(inconsistencias)
df_inconsistencias.insert(0, "id", range(1, len(df_inconsistencias) + 1))
df_inconsistencias.to_csv(PASTA_DOCS / "registro_inconsistencias.csv", index=False)

print(df_inconsistencias["impacto"].value_counts())
df_inconsistencias

impacto
baixo    7
médio    4
alto     1
Name: count, dtype: int64


,id,fonte,local,problema,evidencia,impacto,decisao
0,1,Anuário 2026,"Tabelas 17, 20 e 23",As variações 2024/2025 usam bases diferentes p...,Reconciliação (seção 2): a soma dos volumes de...,alto,"Não comparar o +21,01% do puro malte com a que..."
1,2,Anuário 2026,Tabela 17 × Anuário 2025,A retificação de 2024 não foi uniforme: Nordes...,"Reconciliação (seção 2), coluna dif_pct_vs_pub...",médio,Volumes regionais de 2024 publicados no Anuári...
2,3,Anuário 2025,Tabela 17,Volume do Centro-Oeste impresso como 1.074.137...,"Só com 1.047.137.071,79 L as regiões somam o t...",baixo,"Usado 1.047.137.071,79 L, com observação na co..."
3,4,Anuário 2025,Tabela 18,A coluna Produção Média não corresponde a volu...,A média recalculada coincide com os valores ci...,baixo,Coluna impressa mantida; análises usam `produc...
4,5,Anuário 2026,Gráfico 2,"A legenda indica variação 2023/2024, mas os va...",São Paulo: 427 cervejarias no Anuário 2025 e 4...,baixo,"Variação tratada como 2024/2025, com observaçã..."
5,6,Anuário 2026,Texto do Gráfico 8 (p. 35),O superávit de 2025 aparece como US$ 195.031.0...,Tabela 6 − Tabela 7 = US$ 208.937.248. O cresc...,baixo,Superávit de 2025 calculado a partir das tabel...
6,7,Anuário 2026,Gráficos 10 e 12,Títulos trocados. O Gráfico 10 ('importadores ...,"O Gráfico 10 lista Alemanha, Argentina e Urugu...",baixo,Gráficos não usados; comércio exterior vem do ...
7,8,Anuário 2026,Texto da Tabela 7 (p. 34) e Tabela 10,Erros de digitação: volume de 2024 escrito '7....,"Comparação entre texto, tabelas e Anuário 2025.",baixo,Valores das tabelas prevalecem sobre o texto.
8,9,Anuário 2026,Tabelas 7 e 13,Importação de cerveja em 2025: volume +251% co...,"O Comex Stat confirma os 26,3 milhões de litro...",médio,Investigar na fase 4: localizar nos dados mens...
9,10,Comex Stat,Definição de cerveja,O Anuário considera cerveja a soma dos NCM 220...,Só com os dois códigos os totais de 2024 batem...,médio,Cerveja = 2203.00.00 + 2202.91.00. Em 2015 e 2...


## 5. Dicionário de dados

A descrição de cada arquivo está em `docs/dicionario_de_dados.md`. 

## Conclusão

**Os dados estão consistentes para a análise, com uma ressalva importante.**

- Todas as conferências de soma passam: regiões, tipos de cerveja e faixas de abrangência fecham com o total nacional nos dois anos.
- A única conferência reprovada é a do superávit da balança comercial, por um erro no texto do Anuário 2026 (registro 6).
- **A ressalva:** o Anuário 2026 compara 2025 com duas bases diferentes de 2024. A queda de 8,85% usa a base retificada; o crescimento de 21% do puro malte usa a original. Por isso, o indicador de premiumização usado no projeto é a **participação do puro malte no volume, que subiu de 24,7% para 29,2%**. Com a retificação distribuída proporcionalmente, o crescimento estimado do puro malte é de cerca de 8%.

As 12 inconsistências encontradas e as decisões tomadas estão em `docs/registro_inconsistencias.csv`.